# Weighted event sampling to `scipp`

McStas event files contain weighted events. This notebook focuses on converting those weighted events to a fixed number of normal, unit-weight events while exporting to the simple `scipp` representation.

## Imports

In [ ]:
from pathlib import Path

import numpy as np

import mcstastox
from read_example import make_instrument

## Run a small McStas simulation

As in the other user-guide notebooks, use `read_example` to create a small instrument and generate a McStas NeXus event file. The `Read` interface takes the output folder, while McStas writes the file as `mccode.h5` inside that folder.

In [ ]:
instr = make_instrument()
instr.set_parameters(wavelength=1.8, delta_wavelength=1.3)
instr.settings(ncount=1E6)
data = instr.backengine()
data_folder = Path(data[0].original_data_location)
data_folder

## Inspect the weighted `scipp_simple` export

`export_scipp_simple` returns one `scipp.DataArray`. Before sampling, its data values are the McStas event weights (`p`). Reading in chunks avoids assembling the complete event file while it is loaded.

In [ ]:
chunk_size = 1_000
wavelength = mcstastox.Variable(
    coord_name="sim_wavelength",
    variable_name="L",
    unit="angstrom",
)

with mcstastox.Read(data_folder) as loaded_data:
    weighted = loaded_data.export_scipp_simple(
        source_name="source",
        sample_name="sample_position",
        component_name="Square_1",
        extra_variables=wavelength,
        chunk_size=chunk_size,
    )

weighted

In [ ]:
weighted.sizes, weighted.values[:5], weighted.coords["sim_wavelength"].values[:5]

## Configure sampling

`SamplingSettings` uses weighted reservoir sampling with replacement. The result contains exactly `n_samples` events, so an input event may appear more than once. A seed makes the result reproducible.

Here, the **input stream** is the sequence of event records read from the selected McStas component. Each record contains the weight (`p`), time (`t`), pixel ID (`id`), and any requested extra variables. With chunked loading, the stream is the same logical sequence as a full read: rows are visited in file order, and selected components are visited in component order.

By default, the reservoir is returned in its internal sampling order, so the output order is not meaningful. Set `ordered=True` to retain each sampled record's position in the input stream and sort the result back into that order. This changes only the arrangement of the sampled records, not which weighted sample is selected.

In [ ]:
sampling = mcstastox.SamplingSettings(
    n_samples=1_000,
    seed=42,
    ordered=True,
)
sampling

## Export sampled normal events

Pass the settings with the `sampling=` keyword. Sampling happens while the input chunks are read, and the exported data values are all one count. Extra event coordinates are sampled with the same events and remain aligned.

In [ ]:
with mcstastox.Read(data_folder) as loaded_data:
    sampled = loaded_data.export_scipp_simple(
        source_name="source",
        sample_name="sample_position",
        component_name="Square_1",
        extra_variables=wavelength,
        chunk_size=chunk_size,
        sampling=sampling,
    )

sampled

In [ ]:
assert sampled.sizes["events"] == sampling.n_samples
np.testing.assert_array_equal(sampled.values, np.ones(sampling.n_samples))

sampled.sizes, sampled.values[:5], sampled.coords["sim_wavelength"].values[:5]

The same settings can be used to reproduce the same event selection. Change `seed` to obtain another statistically weighted sample.

In [ ]:
with mcstastox.Read(data_folder) as loaded_data:
    sampled_again = loaded_data.export_scipp_simple(
        source_name="source",
        sample_name="sample_position",
        component_name="Square_1",
        extra_variables=wavelength,
        chunk_size=chunk_size,
        sampling=sampling,
    )

np.testing.assert_array_equal(sampled.values, sampled_again.values)
np.testing.assert_array_equal(
    sampled.coords["t"].values, sampled_again.coords["t"].values
)
print("Sampling is reproducible with the same seed.")